In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catlog_name}.{bronze_schema}.constructors"
silver_table = f"{catlog_name}.{silver_schema}.constructors"

In [0]:
constructors_df = spark.read.table(bronze_table)

In [0]:
display(constructors_df)

In [0]:
from pyspark.sql import functions as F

In [0]:
constructors_selected_df = constructors_df.select(
    F.col("constructorId"),
    F.col("name"),
    F.col("nationality"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)

In [0]:
constructors_renamed_df = (constructors_selected_df
                           .withColumnsRenamed({
                               "constructorId": "constructor_id",
                               "name": "constructor_name"
                           }))

In [0]:
display(constructors_renamed_df)

In [0]:
constructor_distinct_df = constructors_renamed_df.dropDuplicates()

In [0]:
display(constructor_distinct_df)

In [0]:
constructors_final_df = (constructor_distinct_df
                        .withColumn("nationality", F.initcap(F.col("nationality")))
                        )

In [0]:
display(constructors_final_df)

In [0]:
(
    constructors_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
    
)

In [0]:
display(spark.table(silver_table))